# Creation of Gold table

This script generates a gold table, containing data in a format optimized for Machine Learning.

In a real scenario, we would apply filters by quality, as well as imputation and scaling. For this case, we just use a filter by variance>0.01 across all datasets.

In [0]:
from pyspark.sql.functions import col

# Step 1: Load probe-level stats and filter for high-variance, shared probes
stats_df = spark.table("silver.methylation.methylation_beta_stats")

filtered_probes_df = stats_df.filter(
    (col("dataset_count") == 2) &
    (col("beta_variance") > 0.05)
).select("probe_id")

# Step 2: Load long-format methylation table and join on selected probes
beta_df = spark.table("silver.methylation.methylation_beta")

filtered_df = beta_df.join(filtered_probes_df, on="probe_id", how="inner") \
    .select("sample_id", "probe_id", "beta")

# Step 3: Pivot to wide format: rows = samples, columns = probe_ids
pivot_df = filtered_df.groupBy("sample_id").pivot("probe_id").agg({"beta": "first"})

# Step 4: Write to Delta table in gold layer
pivot_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold.methylation.methylation_beta_005")

print("✅ Gold table created: gold.methylation.methylation_beta_005")


In [0]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: Load gold table into pandas
df = spark.table("gold.methylation.methylation_beta_001").toPandas()
df.set_index("sample_id", inplace=True)

# Step 2: Join sample group info
# Assumes sample_group info exists in silver table
sample_meta = spark.table("silver.methylation.methylation_beta") \
    .select("sample_id", "sample_group") \
    .distinct().toPandas().set_index("sample_id")

df["sample_group"] = sample_meta.loc[df.index, "sample_group"].values

# Step 3: Standardize features and run PCA
X = StandardScaler().fit_transform(df.drop(columns="sample_group"))
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Step 4: Plot PCA
pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"], index=df.index)
pca_df["sample_group"] = df["sample_group"].values

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="sample_group", palette="Set2", s=50)
plt.title("PCA of Gold Methylation Table (Probes > 0.01 variance)")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.grid(True)
plt.legend(title="Sample Group")
plt.tight_layout()
plt.show()
